In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc('font', size=18)
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

import xarray as xr
import cartopy.crs as ccrs
data_proj = ccrs.PlateCarree()

def errorband(x,y,yerr, ax=None, band_alpha=0.5, color=None, **kwargs):
    if ax is None:
        ax = plt
    line = ax.plot(x, y, color=color, **kwargs)
    if color is None:
        color = plt.gca().line[-1].get_color()
    shadey = ax.fill_between(x, y-yerr, y+yerr, color=color, alpha=band_alpha)
    return line, shadey

In [ ]:
# load data
ds_proj = xr.open_dataset('M.nc')
ds_test = xr.open_dataset('test_set.nc')

In [ ]:
# plot figure

plt.close(1)
fig = plt.figure(num=1, figsize=(20, 10))


# plot geoplots

LON, LAT = np.meshgrid(ds_proj.lon, ds_proj.lat)
projs = np.concatenate([ds_proj.GA.data, ds_proj.IINN.data], axis=-1)

projections = [ccrs.Orthographic(central_latitude=90), ccrs.PlateCarree()]
mx = [7,15]
cmaps=['RdBu_r', 'BrBG']
titles=['Geopotential height', 'Soil moisture']
extent_sm = (-5, 10, 39, 55)

for i in range(4):
    ax = fig.add_subplot(241 + i, projection=projections[i%2])

    _mx = mx[i%2]
    _norm = matplotlib.colors.TwoSlopeNorm(vcenter=0., vmin=-_mx, vmax=_mx)

    im = ax.pcolormesh(LON, LAT, projs[...,i], transform=data_proj, cmap=cmaps[i%2], norm=_norm)
    ax.coastlines()
    ax.set_title(titles[i%2])
    plt.colorbar(im, extend='both')
    
    if i%2:
        ax.set_extent(extent_sm)


# plot projected space

axs = [fig.add_subplot(223), fig.add_subplot(224)]

axs[0].scatter(
    ds_test.f_GA,
    ds_test.A,
    marker='.',
    alpha=0.5, color='black', label='data'
)

isort = np.argsort(ds_test.f_GA.values)

errorband(
    ds_test.f_GA.values[isort], 
    ds_test.A_pred_GA_mean.values[isort],
    ds_test.A_pred_GA_std.values[isort],
    color=default_colors[0], label='GA', ax=axs[0]
)

axs[0].set_xlabel(r'$M_\mathrm{GA}\cdot X$', fontdict=dict(size=20))
axs[0].set_ylabel('$A$ [K]', fontdict=dict(size=20))
axs[0].legend()


axs[1].scatter(
    ds_test.f_IINN,
    ds_test.A,
    marker='.',
    alpha=0.5, color='black', label='data',
)

isort = np.argsort(ds_test.f_IINN.values)

errorband(
    ds_test.f_IINN.values[isort],
    ds_test.A_pred_IINN_mean.values[isort],
    ds_test.A_pred_IINN_std.values[isort],
    color=default_colors[1], label='IINN', ax=axs[1]
)

axs[1].set_xlabel(r'$M_\mathrm{IINN}\cdot X$', fontdict=dict(size=20))
axs[1].set_ylabel('$A$ [K]', fontdict=dict(size=20))
axs[1].legend()

fig.suptitle(r'$M_\mathrm{GA}$' + ' '*77 + r'$M_\mathrm{IINN}$', fontsize='x-large')

fig.tight_layout(w_pad=0)

plt.show()